# Core Collaboration Network — Top 1000 GitHub Repos

**Hypothesis (from JIRA analysis):** the _core_ collaboration network of a project is very small relative to its total contributor count.

This notebook fetches the top 1000 GitHub repositories by stars, pulls contributor statistics for each, and computes:

| Metric | What it measures |
|---|---|
| **80/20 core size** | minimum contributors accounting for 80% of all commits |
| **Gini coefficient** | inequality of contribution distribution (0 = equal, 1 = one person does everything) |
| **Top-1 share** | fraction of commits by the single most active contributor |
| **Temporal core** | contributors whose activity windows overlap with at least one other (potential collaborators) |
| **Core fraction** | core size / total contributors |

Results are cached to disk so re-runs are instant.

## Configuration

In [ ]:
GITHUB_TOKEN = ""      # required — 5000 req/hr authenticated vs 60 req/hr anonymous
MAX_REPOS    = 1000    # top N repos by stars
CACHE_FILE   = "top_repos_cache.json"  # intermediate results saved here

In [ ]:
import json, time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from itertools import combinations
from collections import defaultdict

## Fetch top repositories

In [ ]:
def make_headers(token=''):
    h = {"Accept": "application/vnd.github+json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def _get(url, headers, params=None, retries=4):
    delay = 2
    for attempt in range(retries):
        r = requests.get(url, headers=headers, params=params, timeout=30)
        if r.status_code == 200:
            return r
        if r.status_code in (202, 429, 500, 502, 503):
            time.sleep(delay)
            delay *= 2
            continue
        if r.status_code == 403:
            raise PermissionError("Rate limit hit — set GITHUB_TOKEN.")
        return None  # 404, 451, etc.
    return None


def fetch_top_repos(headers, n=1000):
    print(f"Fetching top {n} repos by stars...")
    repos, page = [], 1
    while len(repos) < n:
        r = _get(
            "https://api.github.com/search/repositories",
            headers,
            {"q": "stars:>100", "sort": "stars", "order": "desc", "per_page": 100, "page": page},
        )
        if r is None:
            break
        items = r.json().get('items', [])
        if not items:
            break
        repos.extend(items)
        if "next" not in r.links:
            break
        page += 1
        time.sleep(0.3)
    repos = repos[:n]
    print(f"Got {len(repos)} repos")
    return repos

## Fetch contributor stats (with disk cache)

In [ ]:
def fetch_contributor_stats(owner, repo, headers):
    """Fetch /stats/contributors, retrying on 202 (GitHub is computing)."""
    url = f'https://api.github.com/repos/{owner}/{repo}/stats/contributors'
    r = _get(url, headers)
    return r.json() if r and isinstance(r.json(), list) else None


def fetch_all_stats(repos, headers, cache_file='top_repos_cache.json'):
    cache = {}
    p = Path(cache_file)
    if p.exists():
        cache = json.loads(p.read_text())
        print(f"Cache loaded: {len(cache)} repos")

    results = []
    for i, repo in enumerate(repos):
        full_name = repo['full_name']
        owner, name = full_name.split('/', 1)

        if full_name not in cache:
            cache[full_name] = fetch_contributor_stats(owner, name, headers)
            time.sleep(0.15)

        stats = cache[full_name]
        if stats:
            results.append({
                'repo': full_name,
                'stars': repo['stargazers_count'],
                'forks': repo['forks_count'],
                'language': repo.get('language'),
                'contributors': stats,
            })

        if (i + 1) % 100 == 0:
            p.write_text(json.dumps(cache))  # checkpoint
            print(f"  {i + 1}/{len(repos)} repos processed ({len(results)} with data)")

    p.write_text(json.dumps(cache))
    print(f"Done — {len(results)}/{len(repos)} repos returned contributor data")
    return results

## Compute per-repo metrics

In [ ]:
def gini(values):
    """Gini coefficient: 0 = perfect equality, 1 = maximum inequality."""
    v = np.sort(np.array(values, dtype=float))
    n = len(v)
    if n == 0 or v.sum() == 0:
        return 0.0
    ix = np.arange(1, n + 1)
    return float((2 * ix.dot(v) / (n * v.sum())) - (n + 1) / n)


def pareto_core(totals, threshold=0.8):
    """Minimum contributors accounting for `threshold` fraction of total commits."""
    desc = sorted(totals, reverse=True)
    total = sum(desc)
    if total == 0:
        return 0
    cumsum = 0
    for i, t in enumerate(desc):
        cumsum += t
        if cumsum >= threshold * total:
            return i + 1
    return len(totals)


def activity_window(weeks):
    active = [w['w'] for w in weeks if w.get('c', 0) + w.get('a', 0) + w.get('d', 0) > 0]
    return (min(active), max(active)) if active else None


def temporal_core_size(contributors):
    """Contributors whose activity window overlaps with at least one other (sweep-line O(n log n))."""
    windows = [w for c in contributors for w in [activity_window(c.get('weeks', []))] if w]
    if len(windows) < 2:
        return len(windows)
    windows.sort(key=lambda w: w[0])
    has_overlap = [False] * len(windows)
    active = []  # (end_time, index)
    for i, (start, end) in enumerate(windows):
        active = [(e, j) for e, j in active if e >= start]
        if active:
            has_overlap[i] = True
            for _, j in active:
                has_overlap[j] = True
        active.append((end, i))
    return sum(has_overlap)


def compute_metrics(repo_data):
    records = []
    for repo in repo_data:
        contributors = repo['contributors']
        totals = [c.get('total', 0) for c in contributors if c.get('total', 0) > 0]
        if not totals:
            continue
        total_commits = sum(totals)
        records.append({
            'repo':           repo['repo'],
            'stars':          repo['stars'],
            'forks':          repo['forks'],
            'language':       repo['language'],
            'n_contributors': len(totals),
            'total_commits':  total_commits,
            'gini':           gini(totals),
            'core_80':        pareto_core(totals, 0.8),
            'core_50':        pareto_core(totals, 0.5),
            'top1_share':     max(totals) / total_commits,
            'temporal_core':  temporal_core_size(contributors),
        })
    df = pd.DataFrame(records)
    df['core_fraction'] = df['core_80'] / df['n_contributors']
    return df

## Visualise

Six panels testing the hypothesis that the core collaboration network is small:
1. Distribution of 80/20 core sizes
2. Core size vs total contributors (log-log)
3. Gini coefficient distribution
4. CDF — fraction of repos with core ≤ N
5. Top contributor's share of commits
6. Median core size by programming language

In [ ]:
def plot_stats(df):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        f'Core Collaboration Network — Top {len(df)} GitHub Repos\n'
        f'Median contributors: {df["n_contributors"].median():.0f}  |  '
        f'Median 80/20 core: {df["core_80"].median():.0f}  |  '
        f'Median Gini: {df["gini"].median():.2f}',
        fontsize=13,
    )

    # 1. 80/20 core size histogram
    ax = axes[0, 0]
    ax.hist(df['core_80'].clip(upper=50), bins=40, edgecolor='white')
    ax.axvline(df['core_80'].median(), color='red', linestyle='--',
               label=f"Median: {df['core_80'].median():.0f}")
    ax.set_xlabel('80/20 Core Size (contributors)')
    ax.set_ylabel('Repos')
    ax.set_title('80/20 Core Size Distribution')
    ax.legend()

    # 2. Core vs total (log-log scatter)
    ax = axes[0, 1]
    ax.scatter(df['n_contributors'], df['core_80'], alpha=0.25, s=8, color='steelblue')
    lims = [1, df['n_contributors'].max() * 1.5]
    ax.plot(lims, lims, 'r--', linewidth=0.8, label='core = total')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Total contributors (log)')
    ax.set_ylabel('80/20 Core size (log)')
    ax.set_title('Core vs Total Contributors')
    ax.legend(fontsize=8)

    # 3. Gini coefficient
    ax = axes[0, 2]
    ax.hist(df['gini'], bins=40, edgecolor='white')
    ax.axvline(df['gini'].median(), color='red', linestyle='--',
               label=f"Median: {df['gini'].median():.2f}")
    ax.set_xlabel('Gini Coefficient')
    ax.set_ylabel('Repos')
    ax.set_title('Contribution Inequality (Gini)')
    ax.legend()

    # 4. CDF of core size
    ax = axes[1, 0]
    sorted_core = np.sort(df['core_80'])
    cdf = np.arange(1, len(sorted_core) + 1) / len(sorted_core)
    ax.plot(sorted_core, cdf, linewidth=2)
    for n in [3, 5, 10, 20]:
        pct = (df['core_80'] <= n).mean()
        ax.axvline(n, color='grey', linestyle=':', linewidth=0.8)
        ax.text(n, 0.05, f'{pct:.0%}\n≤{n}', ha='center', fontsize=7)
    ax.set_xlabel('Core Size')
    ax.set_ylabel('Fraction of repos')
    ax.set_title('CDF: Repos with Core ≤ N')
    ax.set_xlim(0, 50)

    # 5. Top-1 contributor share
    ax = axes[1, 1]
    ax.hist(df['top1_share'], bins=40, edgecolor='white')
    ax.axvline(df['top1_share'].median(), color='red', linestyle='--',
               label=f"Median: {df['top1_share'].median():.0%}")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_xlabel('Top Contributor\'s Share')
    ax.set_ylabel('Repos')
    ax.set_title('Top-1 Contributor Dominance')
    ax.legend()

    # 6. Language breakdown
    ax = axes[1, 2]
    lang = (df.groupby('language')['core_80']
            .agg(median='median', count='count')
            .query('count >= 10')
            .sort_values('median'))
    lang['median'].plot(kind='barh', ax=ax)
    ax.set_xlabel('Median 80/20 Core Size')
    ax.set_title('Core Size by Language (n ≥ 10 repos)')

    plt.tight_layout()
    plt.show()

    print('\n=== Summary ===')
    print(f'Repos analysed:             {len(df)}')
    print(f'Median total contributors:  {df["n_contributors"].median():.0f}')
    print(f'Median 80/20 core:          {df["core_80"].median():.0f}')
    print(f'Median core fraction:       {df["core_fraction"].median():.1%}')
    print(f'Median Gini:                {df["gini"].median():.3f}')
    print(f'Median top-1 share:         {df["top1_share"].median():.1%}')
    print(f'Repos where core ≤ 3:       {(df["core_80"] <= 3).mean():.1%}')
    print(f'Repos where core ≤ 5:       {(df["core_80"] <= 5).mean():.1%}')
    print(f'Repos where core ≤ 10:      {(df["core_80"] <= 10).mean():.1%}')

## Run

In [ ]:
HEADERS = make_headers(GITHUB_TOKEN)

repos    = fetch_top_repos(HEADERS, n=MAX_REPOS)
raw_data = fetch_all_stats(repos, HEADERS, cache_file=CACHE_FILE)

df = compute_metrics(raw_data)
print(df.describe())

plt.style.use('fivethirtyeight')
plot_stats(df)